# accel-sim silicon anchor — chain generalization test

The chain-depth anchor found a clean fixed-overhead + per-layer-marginal
model for a 768×768 chain at 8192 tokens (R²=0.97) — but applying that same
fix to other shapes/token-counts made 2 of 4 configs *worse*. Digging in:
`wide_shallow`'s wgrad already has full occupancy under the old model (no
penalty to relieve), yet it was still measured ~2.8× faster than
predicted — proof of a **separate, occupancy-independent chain effect**.
And `small_batch` (1024 tokens) broke because a fixed-ms overhead calibrated
at 8192 tokens swamps a config whose whole measured time is under 1ms.

This notebook isolates both remaining unknowns in one combined design, on
a shape (2048×2048) with **occupancy=1.0 for both dgrad and wgrad** — so
occupancy never confounds the result:

1. **Depth sweep** (1/2/4/8) at fixed 8192 tokens — does the fixed-overhead
   effect exist even with no occupancy issue at all?
2. **Token-count sweep** (512/1024/2048/4096/8192/16384) at fixed depth=4 —
   does the overhead scale with token count, or is it genuinely fixed?
3. **One repeated config** (depth=4, 8192 tokens, run again last) — bounds
   how much of any "doesn't generalize" result is real physics vs. Colab
   session/tenancy noise.

**Before running:** `Runtime > Change runtime type > T4 GPU`, then
`Runtime > Run all`.

Writes `chain_generalize_profile.json`, prints it, and auto-downloads it.
Bring that file back and run:

```bash
python validate/silicon/compare_chain_generalize.py chain_generalize_profile.json
```


In [ ]:
# ---- config (edit if you want) ---------------------------------------------
ITERS  = 50
WARMUP = 15
OUT    = "chain_generalize_profile.json"

SHAPE = (2048, 2048)   # occupancy=1.0 for dgrad AND wgrad -- no confound

CONFIGS = [
    ("depth1_m8192",  1, 8192),
    ("depth2_m8192",  2, 8192),
    ("depth4_m8192",  4, 8192),
    ("depth8_m8192",  8, 8192),
    ("depth4_m512",   4, 512),
    ("depth4_m1024",  4, 1024),
    ("depth4_m2048",  4, 2048),
    ("depth4_m4096",  4, 4096),
    ("depth4_m16384", 4, 16384),
    ("repeat_depth4_m8192", 4, 8192),   # noise check -- run last
]


In [ ]:
import torch
assert torch.cuda.is_available(), "no CUDA device -- Runtime > Change runtime type > T4 GPU"

device = torch.device("cuda")
dtype = torch.float16
gpu = torch.cuda.get_device_name(0)
print(f"GPU: {gpu}   dtype=fp16   shape={SHAPE}   iters={ITERS} (+{WARMUP} warmup)")


In [ ]:
import statistics
import torch.nn as nn

def bench(depth, M, iters, warmup):
    layers = [nn.Linear(*SHAPE, bias=False).to(device=device, dtype=dtype)
              for _ in range(depth)]
    params = [p for l in layers for p in l.parameters()]
    opt = torch.optim.Adam(params, lr=1e-4)
    x = torch.randn(M, SHAPE[0], device=device, dtype=dtype, requires_grad=True)

    fwd, bwd, optt = [], [], []
    for i in range(warmup + iters):
        ev = [torch.cuda.Event(enable_timing=True) for _ in range(5)]
        ev[0].record()
        out = x
        for l in layers:
            out = l(out)
        ev[1].record()
        loss = out.float().square().mean()
        ev[2].record()
        loss.backward()
        ev[3].record()
        opt.step()
        opt.zero_grad(set_to_none=True)
        x.grad = None
        ev[4].record()
        torch.cuda.synchronize()
        if i >= warmup:
            fwd.append(ev[0].elapsed_time(ev[2]))
            bwd.append(ev[2].elapsed_time(ev[3]))
            optt.append(ev[3].elapsed_time(ev[4]))

    def stat(v):
        return {"mean_ms": statistics.fmean(v),
                "std_ms": statistics.pstdev(v) if len(v) > 1 else 0.0}
    return {"forward": stat(fwd), "backward": stat(bwd), "optimizer": stat(optt)}


In [ ]:
results = {}
for name, depth, M in CONFIGS:
    r = bench(depth, M, ITERS, WARMUP)
    results[name] = {"depth": depth, "tokens": M, **r}
    print(f"  {name:20s} fwd {r['forward']['mean_ms']:8.3f}  "
          f"bwd {r['backward']['mean_ms']:8.3f}  "
          f"opt {r['optimizer']['mean_ms']:6.3f} ms")


In [ ]:
import json, platform

out = {
    "gpu": gpu, "torch": torch.__version__, "cuda": torch.version.cuda,
    "dtype": "float16", "shape": SHAPE, "iters": ITERS, "warmup": WARMUP,
    "results": results, "host": platform.platform(),
}
with open(OUT, "w") as f:
    json.dump(out, f, indent=2)

print(f"\n===== {OUT} (copy this back if the download fails) =====\n")
print(json.dumps(out, indent=2))

try:
    from google.colab import files
    files.download(OUT)
except Exception as e:
    print(f"\n(auto-download unavailable: {e} -- grab {OUT} from the Files sidebar)")
